# 06 - Submission
Chọn mô hình tốt nhất từ kết quả CV, fit lại trên toàn bộ train và tạo submission thật.


## Chuẩn bị


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

OUTPUT = PROJECT_ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)



## Hàm: `log`


In [2]:
def log(message):
    print(message)
    with (OUTPUT / "train_log.txt").open("a", encoding="utf-8") as file:
        file.write(f"{message}\n")

import pandas as pd
from notebook_loader import run_notebook_code
from lab03_plots import plot_price_distribution
run_notebook_code(PROJECT_ROOT / "house_price_pipeline.ipynb", globals())
run_notebook_code(PROJECT_ROOT / "pytorch_mlp.ipynb", globals())



## Hàm: `fit_full_mlp`


In [3]:
def fit_full_mlp(train: pd.DataFrame, test: pd.DataFrame, random_state: int, quick: bool):
    # Huấn luyện lại MLP tốt nhất trên toàn bộ tập train trước khi dự đoán test.
    x_train = make_feature_frame(train, "engineered")
    x_test = make_feature_frame(test, "engineered")
    y = np.log1p(train[TARGET].astype(float).to_numpy())
    pipe = Pipeline(
        [
            ("preprocess", make_preprocessor(x_train)),
            (
                "model",
                TorchMLPRegressor(
                    max_epochs=110 if quick else 340,
                    patience=18 if quick else 35,
                    random_state=random_state,
                ),
            ),
        ]
    )
    pipe.fit(x_train, y)
    pred = np.expm1(pipe.predict(x_test)).clip(min=0)
    return pipe, pred





## Chạy notebook


In [4]:
result_files = [path for path in [OUTPUT / "ml_results.csv", OUTPUT / "mlp_results.csv"] if path.exists()]
if not result_files:
    raise FileNotFoundError("Hãy chạy notebook 03 hoặc 04 trước.")

best = pd.concat([pd.read_csv(path) for path in result_files]).sort_values("log_rmse_mean").iloc[0]
log(f"[7] Creating submission.csv với {best['experiment']}...")
train, test = load_competition_data(PROJECT_ROOT / "data")
if best["model"] == "pytorch_mlp":
    pipeline, prediction = fit_full_mlp(train, test, random_state=42, quick=True)
    model_path = OUTPUT / "best_model_mlp.joblib"
else:
    pipeline, prediction = fit_full_sklearn(
        train, test, best["feature_profile"], best["model"], bool(best["remove_outliers"]), quick=True
    )
    model_path = OUTPUT / "best_model.joblib"
save_pipeline(pipeline, model_path)
submission = pd.DataFrame({"Id": test["Id"].astype(int), "SalePrice": prediction})
submission.to_csv(OUTPUT / "submission.csv", index=False)
plot_price_distribution(train[TARGET], OUTPUT / "submission_price_distribution.png", submission["SalePrice"])
log("Done.")


[7] Creating submission.csv với engineered_gradient_boosting...
Done.
